## Nesse notebook farei limpeza e feature engeneering para preparar o dataset para o modelo de ML

### Estamos preparando os dados para definir que vai churnar.

In [64]:
import pandas as pd 
import numpy as np

PATH = "../data/raw/churn-prediction.csv"

In [65]:
df = pd.read_csv(PATH)

df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Irei remover features que nao ajudam a prever o churn. E criar novas que fazem sentido para prever.

### A coluna costumer_id sera removida, enquanto as colunas contry e gender que sao categoricas transformarei em numerica

In [66]:
df = df.drop("customer_id", axis=1)

In [67]:
df.head()

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Para gender irei utilizar 0 para Male e 1 para Female

In [68]:
# Converter gender: 0 para Male, 1 para Female
df['gender'] = (df['gender'] == 'Female').astype(int)

df.head()

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,France,1,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,1,41,1,83807.86,1,0,1,112542.58,0
2,502,France,1,42,8,159660.80,3,1,0,113931.57,1
3,699,France,1,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,1,43,2,125510.82,1,1,1,79084.10,0


### Pro pais usarei OneHotEncoder

In [69]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder( sparse_output=False)
country_encoded = ohe.fit_transform(df[['country']])
country_columns = ohe.get_feature_names_out(['country'])
df_encoded = pd.DataFrame(country_encoded, columns=country_columns, index=df.index)
df = pd.concat([df.drop('country', axis=1), df_encoded], axis=1)

df.head()

,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_France,country_Germany,country_Spain
0,619,1,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,1,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,1,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,1,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,1,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


### Criarei novas features para auxiliar na previsao
- has_balance
- dead_client
- age_group
- product_group

#### has_balance -> Verifica se o cliente tem ou nao saldo, sendo 0 para nao tem e 1 para tem

In [70]:
df["has_balance"] = (df["balance"] > 0).astype(int)

df.head()

,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_France,country_Germany,country_Spain,has_balance
0,619,1,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,0
1,608,1,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,1
2,502,1,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,1
3,699,1,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,0
4,850,1,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,1


#### dead_client -> Verifica se o cliente  e inativo e nao tem saldo

In [71]:
df["dead_client"] = ((df["balance"] == 0) & (df["active_member"] == 0)).astype(int)

df.head()

,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_France,country_Germany,country_Spain,has_balance,dead_client
0,619,1,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,0,0
1,608,1,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,1,0
2,502,1,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,1,0
3,699,1,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,0,1
4,850,1,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,1,0


#### age_group -> Separacao das idades em quartis para evitar o uso de cortes arbitrarios.

In [72]:
df["age_group"] = pd.qcut(df["age"], q=4, labels=False)

df.head()

,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_France,country_Germany,country_Spain,has_balance,dead_client,age_group
0,619,1,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,0,0,2
1,608,1,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,1,0,2
2,502,1,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,1,0,2
3,699,1,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,0,1,2
4,850,1,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,1,0,2


#### product_group -> Agrupei baseado no numero de produtos(servicos),alto e baixo, para refletir o engajamento. Clientes com 3 ou mais produtos mostraram um comportamento similar, viraram alto (1) e os baixos(0), mantendo assim o padrao visto na EDA

In [73]:
df["product_group"] = (df["products_number"] >= 3).astype(int)

In [74]:

df.head()

,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_France,country_Germany,country_Spain,has_balance,dead_client,age_group,product_group
0,619,1,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,0,0,2,0
1,608,1,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,1,0,2,0
2,502,1,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,1,0,2,1
3,699,1,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,0,1,2,0
4,850,1,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,1,0,2,0


### Salvando o dataset processado

In [75]:
# Salvando o dataset processado
output_path = "../data/processed/churn-processed.csv"
df.to_csv(output_path, index=False)
print(f"Dataset salvo em: {output_path}")

Dataset salvo em: ../data/processed/churn-processed.csv
